In [1]:
import importlib

In [2]:


import pandas as pd
import data_loader
import user_profile as user_pf


In [3]:
# Change this to True if you are running on Google Colab
RUNNING_ON_COLAB = False

# Constants
USER_ID = 999999

In [4]:
if RUNNING_ON_COLAB:
    data_loader.mount_drive()

In [5]:
# Load the data
films_df = data_loader.load_movies()
ratings_df = data_loader.load_ratings()
credits_df = data_loader.load_credits()

/Users/Cathal/Rec-Genie/rec-sys/data_loader.py:15: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(dataset_path + "/movies_metadata.csv", usecols=['id', 'title', 'release_date', 'genres', 'popularity', 'vote_average', 'vote_count'])


In [6]:
print("Data dimensions:")
print("films_df: ", films_df.shape)
print("ratings_df: ", ratings_df.shape)
print("credits_df: ", credits_df.shape)

Data dimensions:
films_df:  (45466, 7)
ratings_df:  (32000204, 3)
credits_df:  (45476, 3)


In [7]:
credits_df.shape

(45476, 3)

In [8]:
films_df.columns

Index(['genres', 'id', 'popularity', 'release_date', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [9]:
# Find film_df entry 100 - Lock Stock and Two Smoking Barrels
films_df[films_df['id'] == 100]

,genres,id,popularity,release_date,title,vote_average,vote_count


In [10]:
# Drop the weird film entry
films_df = films_df.drop(35587)

In [11]:
import preprocessing as pre

In [12]:
# Data preprocessing
# One-hot encode the genres
ohe_films_df, genre_list_mlb = pre.one_hot_encode_genres(films_df)

In [13]:
ohe_films_df.columns

Index(['id', 'popularity', 'release_date', 'title', 'vote_average',
       'vote_count', 'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Family', 'Fantasy', 'Foreign', 'History',
       'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie',
       'Thriller', 'War', 'Western'],
      dtype='object')

In [14]:
# Gather info on directors and cast
credits_df = pre.condense_credits(credits_df)

In [15]:
print(type(ohe_films_df), type(credits_df))  # Debugging line

<class 'pandas.core.frame.DataFrame'> <class 'pandas.core.frame.DataFrame'>


In [16]:

# Tidy the noise and merge the credits' metadata with the films DataFrame
films_df = pre.data_tidying(ohe_films_df, credits_df)
print(films_df.columns)


Index(['id', 'popularity', 'release_date', 'title', 'vote_average',
       'vote_count', 'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Family', 'Fantasy', 'Foreign', 'History',
       'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie',
       'Thriller', 'War', 'Western', 'cast_info', 'director_info'],
      dtype='object')


In [17]:
# Generate the user profile
user_ratings_df = user_pf.load_user_ratings()
user_profile = user_pf.create_user_profile(USER_ID, films_df, user_ratings_df, genre_list_mlb)

In [18]:
# Append the user profile to the ratings DataFrame
ratings_df = pd.concat([ratings_df, user_ratings_df], ignore_index=True)

In [19]:
import hybrid_recommender as hyb
importlib.reload(hyb)

<module 'hybrid_recommender' from '/Users/Cathal/Rec-Genie/rec-sys/hybrid_recommender.py'>

In [20]:

# Generate recommendations
recommendations = hyb.hybrid_recommend(999999, user_profile, films_df, credits_df, ratings_df, genre_list_mlb)


Numba is using threading layer workqueue - consider TBB
found 1 potential runtime problems - see https://boi.st/lkpy-perf


User-User algorithm set up!


In [21]:
len(recommendations)

400

In [22]:
score_breakdown = hyb.score_breakdown(films_df, recommendations)

In [23]:
score_breakdown.head(50)sadsa

,id,title,release_date,vote_average,vote_count,score,cast_score,director_score,genre_score,user_user_score
5252,1894,Star Wars: Episode II - Attack of the Clones,2002-05-15,6.4,4074.0,10.323623,7.98,0.00,5.8320,3.104663
10138,272,Batman Begins,2005-06-10,7.5,7511.0,9.252858,5.40,0.00,7.5820,3.349898
11104,1164,Babel,2006-09-08,6.9,1083.0,8.733337,3.21,0.00,10.1580,3.642097
1310,2212,Nightwatch,1997-01-31,6.2,104.0,8.596010,4.74,0.00,4.9164,3.901418
12331,4787,Cassandra's Dream,2007-06-18,6.1,219.0,8.584150,4.74,0.00,6.6820,3.395190
10584,1420,Breakfast on Pluto,2005-09-03,7.0,77.0,8.547969,4.08,0.00,7.2870,3.651609
12615,8681,Taken,2008-02-18,7.2,4444.0,8.470072,4.08,0.00,6.6540,3.750952
278,1945,Nell,1994-12-23,6.1,128.0,8.344134,4.08,0.00,7.7310,3.323454
9557,2567,The Aviator,2004-12-17,7.0,1526.0,8.335127,3.63,0.00,10.1580,2.949887
4186,824,Moulin Rouge!,2001-03-09,7.4,1348.0,8.324154,4.74,0.00,3.4580,4.037914


In [24]:
# weights = {
#     'cast_ft_weight': 0.3,
#     'director_ft_weight': 0.3,
#     'genre_ft_weight': 0.15,
#     'user_user_weight': 1,
#     'content_weight': 0.7,
#     'collab_weight': 0.3
# }